In [1]:
import pandas as pd
import numpy as np

In [6]:
CSV_PATH = 'data/연도별 물동량.csv'
df = pd.read_csv(CSV_PATH)
df

,년도,항구분,총계,외항소계,외항입항,외항출항,외항입항환적,외항출항환적,내항연안화물
0,1993,북항,69468242,57526251,26152577,31373674,0,0,11941991
1,1994,북항,81680460,67960869,33397436,34563433,0,0,13719591
2,1994,감천,3429201,1740411,1662278,78133,0,0,1688790
3,1995,북항,93438046,76290585,38447263,37843322,0,0,17147461
4,1995,감천,3520227,2067929,1966863,101066,0,0,1452298
...,...,...,...,...,...,...,...,...,...
88,2023,감천,9514187,5216822,3711312,1263459,154877,87174,4297365
89,2023,신항,302088186,300500352,28154479,61677733,105299272,105368868,1587834
90,2024,북항,463488214,451052426,70487383,92766297,145029464,142769282,12435788
91,2024,감천,9601761,5459871,3499340,1771395,124721,64415,4141890


In [7]:
df.head

<bound method NDFrame.head of       년도 항구분         총계       외항소계      외항입항      외항출항     외항입항환적     외항출항환적  \
0   1993  북항   69468242   57526251  26152577  31373674          0          0   
1   1994  북항   81680460   67960869  33397436  34563433          0          0   
2   1994  감천    3429201    1740411   1662278     78133          0          0   
3   1995  북항   93438046   76290585  38447263  37843322          0          0   
4   1995  감천    3520227    2067929   1966863    101066          0          0   
..   ...  ..        ...        ...       ...       ...        ...        ...   
88  2023  감천    9514187    5216822   3711312   1263459     154877      87174   
89  2023  신항  302088186  300500352  28154479  61677733  105299272  105368868   
90  2024  북항  463488214  451052426  70487383  92766297  145029464  142769282   
91  2024  감천    9601761    5459871   3499340   1771395     124721      64415   
92  2024  신항  336726664  334921716  32335988  63428330  120568494  118588904   

      내항연

In [8]:
df.shape

(93, 9)

In [9]:
df.columns.to_list()

['년도', '항구분', '총계', '외항소계', '외항입항', '외항출항', '외항입항환적', '외항출항환적', '내항연안화물']

In [13]:
df.dtypes

년도        int64
항구분         str
총계        int64
외항소계      int64
외항입항      int64
외항출항      int64
외항입항환적    int64
외항출항환적    int64
내항연안화물    int64
dtype: object

In [14]:
pd.DataFrame({
    '자료형' : df.dtypes.astype('str'),
    '비결측 수' : df.notna().sum(),
    '결측 수' : df.isna().sum(),
    '결측 률(%)' : df.isna().mean()*100,
    '고유값 수' :df.nunique(dropna=True)
            })

,자료형,비결측 수,결측 수,결측 률(%),고유값 수
년도,int64,93,0,0.0,32
항구분,str,93,0,0.0,3
총계,int64,93,0,0.0,93
외항소계,int64,93,0,0.0,92
외항입항,int64,93,0,0.0,92
외항출항,int64,93,0,0.0,90
외항입항환적,int64,93,0,0.0,71
외항출항환적,int64,93,0,0.0,72
내항연안화물,int64,93,0,0.0,93


In [15]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
년도,93.0,2.009344e+03,8.631961e+00,1993.0,2003.0,2009.0,2017.0,2024.0
총계,93.0,1.343859e+08,1.488320e+08,591.0,9882669.0,69468242.0,243564954.0,468760569.0
외항소계,93.0,1.283562e+08,1.460214e+08,0.0,5926059.0,57526251.0,232130080.0,457533491.0
외항입항,93.0,2.468115e+07,2.426689e+07,0.0,3841529.0,14007530.0,43048407.0,82551755.0
외항출항,93.0,3.325789e+07,3.440597e+07,0.0,1533499.0,27746295.0,61599600.0,107605724.0
외항입항환적,93.0,3.558148e+07,4.651894e+07,0.0,3554.0,1863444.0,72353399.0,145029464.0
외항출항환적,93.0,3.483566e+07,4.590788e+07,0.0,1900.0,1901184.0,70460124.0,142769282.0
내항연안화물,93.0,6.029687e+06,5.738788e+06,591.0,1099104.0,3762389.0,11941991.0,20477638.0


In [58]:
df_yearly = (
    df.groupby(['년도', '항구분'])[
        ['총계', '외항입항', '외항출항', '외항입항환적', '외항출항환적']
    ]
    .sum()
    .reset_index()
)

df_yearly = df_yearly.sort_values(by=['항구분', '년도']).reset_index(
    drop=True
)

df_yearly['환적_총량'] = df_yearly['외항입항환적'] + df_yearly['외항출항환적']
df_yearly['순수수출입_총량'] = df_yearly['외항입항'] + df_yearly['외항출항']
df_yearly['환적_비중(%)'] = (df_yearly['환적_총량'] / df_yearly['총계']) * 100

df_yearly['총계_증가량'] = df_yearly.groupby('항구분')['총계'].diff()
df_yearly['환적_증가량'] = df_yearly.groupby('항구분')['환적_총량'].diff()

df_yearly['환적_성장기여율(%)'] = (
    df_yearly['환적_증가량'] / df_yearly['총계_증가량']
) * 100

result = (
    df_yearly[
        ['년도', '항구분', '총계', '환적_총량', '환적_비중(%)', '환적_성장기여율(%)']
    ]
    .sort_values(by=['년도', '항구분'])
    .tail(20)
)

result

,년도,항구분,총계,환적_총량,환적_비중(%),환적_성장기여율(%)
56,2018,북항,461461501,262462842,56.876433,85.796500
76,2018,신항,305888919,195504468,63.913550,75.676535
25,2019,감천,8715638,258455,2.965417,43.260964
57,2019,북항,468760569,269979175,57.594259,102.976613
77,2019,신항,319416992,209655835,65.637033,104.607412
26,2020,감천,8186236,196423,2.399430,11.717372
58,2020,북항,410953663,248791440,60.540022,36.652602
78,2020,신항,276661237,192885777,69.719119,39.222926
27,2021,감천,10361268,573258,5.532701,17.325492
59,2021,북항,442558714,266770320,60.279080,56.886097


1. ## 주요 항구별 특징 및 비중

신항 : 환적물동량 비중이 63.9% ~ 71.0% 수준으로 3개 항구 중 가장 높으며, 전체 물동량의 대다수를 처리하는 핵심 거점입니다.

북항 : 환적 비중이 57.5% ~ 62.1% 수준을 일정하게 유지하며 신항과 함께 환적 화물을 분산 처리하고 있습니다.

감천 : 환적 비중이 1.9% ~ 5.5% 수준으로 매우 낮아, 환적보다는 수출입/수산물 화물 위주의 항구임을 보여줍니다.

1. ## 환적 비중(%)의 장기적 상승 추세

신항의 구조적 성장: 2018년 63.9%에서 2024년 71.0%까지 꾸준히 상승하며 환적 중심항으로서의 입지를 더욱 강화했습니다.

북항의 반등: 2022년 59.3%로 소폭 감소했다가 2024년 62.1%로 다시 회복 및 상승했습니다.

3. ## 코로나19(2020년) 충격과 분기점

2020년 급감: 코로나19 팬데믹 여파로 2020년 전체 물동량과 환적량이 전 항구에서 일시적으로 크게 감소했습니다. (신항 총량: 3.19억 → 2.76억)

2021년 이후 회복: 2021년부터 물동량이 빠르게 회복세를 보였으며, 2024년에는 전 항구가 2018~2019년 수준을 넘어선 최대 물동량을 기록했습니다.

4. ## 환적 성장기여율(%)의 변동성

신항·북항의 높은 기여도: 물동량이 성장하는 시기(2019, 2021~2024년)에는 신항과 북항의 환적 성장기여율이 대부분 80%~106%를 기록하여, 항만 전체 성장을 환적 화물이 주도했음을 의미합니다.

감천항의 특이치 (2024년): 2024년 감천항의 환적 성장기여율이 -60.4%로 급격히 떨어졌는데, 이는 총물동량 증가 대비 환적량이 감소했음을 나타냅니다.